In [2]:
from Lib.data_loader import DataLoader
from Lib.resnet_model import Resnet3DBuilder
from Lib.HistoryGraph import HistoryGraph
import Lib.image as img
from Lib.utils import mkdirs
import os

Using TensorFlow backend.
C:\Anaconda3\envs\HandGestureRecognitionSystem\lib\site-packages\tensorflow\python\framework\dtypes.py:516: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint8 = np.dtype([("qint8", np.int8, 1)])
C:\Anaconda3\envs\HandGestureRecognitionSystem\lib\site-packages\tensorflow\python\framework\dtypes.py:517: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_quint8 = np.dtype([("quint8", np.uint8, 1)])
C:\Anaconda3\envs\HandGestureRecognitionSystem\lib\site-packages\tensorflow\python\framework\dtypes.py:518: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint16 = np.dtype([("qint16", np.int16, 1)])
C:\Anaconda3\e

In [3]:
from math import ceil

In [4]:
from keras.optimizers import SGD

In [5]:
from keras.callbacks import ModelCheckpoint

In [6]:
target_size = (64,96)
nb_frames = 16
skip = 1
nb_classes = 27
batch_size = 64
input_shape = (nb_frames,) + target_size + (3,)

In [7]:
workers = 8
use_multiprocessing = False
max_queue_size = 20

In [8]:
data_root = r"C:\Users\Ganesh Shinde\Downloads\HandGestureDataset"
csv_labels = r"C:\Users\Ganesh Shinde\Downloads\HandGestureDataset\jester-v1-labels.csv"
csv_train = r"C:\Users\Ganesh Shinde\Downloads\HandGestureDataset\jester-v1-train.csv"
csv_val = r"C:\Users\Ganesh Shinde\Downloads\HandGestureDataset\jester-v1-validation.csv"
csv_test = r"C:\Users\Ganesh Shinde\Downloads\HandGestureDataset\jester-v1-test.csv"
data_vid = r"C:\Users\Ganesh Shinde\Downloads\HandGestureDataset\videos"
model_name = "resnet_3d_model"
data_model = r"C:\Users\Ganesh Shinde\Downloads\HandGestureDataset\model"

In [9]:
path_model = os.path.join(data_root, data_model, model_name)
path_vid = os.path.join(data_root, data_vid)
path_labels = os.path.join(data_root, csv_labels)
path_train = os.path.join(data_root, csv_train)
path_val = os.path.join(data_root, csv_val)
path_test = os.path.join(data_root, csv_test)

In [10]:
data = DataLoader(path_vid, path_labels, path_train, path_val, path_test)
mkdirs(path_model, 0o755)
mkdirs(os.path.join(path_model, "graphs"), 0o755)

In [17]:
gen = img.ImageDataGenerator()
gen_train = gen.flow_video_from_dataframe(data.train_df, path_vid, path_classes=path_labels, x_col="video_id", y_col="label",target_size=target_size, batch_size=batch_size, nb_frames=nb_frames, skip=skip, has_ext=True)
gen_val = gen.flow_video_from_dataframe(data.val_df, path_vid, path_classes=path_labels, x_col="video_id", y_col="label",target_size=target_size, batch_size=batch_size, nb_frames=nb_frames, skip=skip, has_ext=True)

Found 118562 video folders belonging to 27 classes.
Found 14787 video folders belonging to 27 classes.


In [12]:
resnet_model = Resnet3DBuilder.build_resnet_101(input_shape, nb_classes, drop_rate = 0.5)
optimizer = SGD(lr=0.01, momentum=0.9, decay=0.0001, nesterov=False)
resnet_model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])
model_file = os.path.join(path_model, 'resnetmodel.hdf5')

In [13]:
model_checkpointer = ModelCheckpoint(model_file, monitor='val_acc', verbose=1, save_best_only=True, mode='max')

In [14]:
history_graph = HistoryGraph(model_path_name = os.path.join(path_model, 'graphs'))

In [15]:
nb_sample_train = data.train_df['video_id'].size
nb_sample_val = data.val_df['video_id'].size

In [ ]:
resnet_model.fit_generator(generator = gen_train,
                          steps_per_epoch = ceil(nb_sample_train/batch_size),
                          epochs = 2,
                          validation_data = gen_val,
                          validation_steps = 20,
                          shuffle = True,
                          verbose = 1,
                          workers = workers,
                          max_queue_size = max_queue_size,
                          use_multiprocessing = use_multiprocessing,
                          callbacks = [model_checkpointer, history_graph])

Epoch 1/2


C:\Anaconda3\envs\HandGestureRecognitionSystem\lib\site-packages\keras\utils\data_utils.py:616: UserWarning: The input 75 could not be retrieved. It could be because a worker has died.
  UserWarning)
